# Performance and Optimization

Caching, explain plans, shuffle, broadcast and performance tradeoffs.

Synthetic retail case study. Run from the repository root.

In [ ]:
from pathlib import Path
import pandas as pd
from pyspark.sql import SparkSession, functions as F, types as T
spark = SparkSession.builder.master("local[2]").appName("SparkLearning").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
root = Path.cwd()
if not (root / "data" / "retail_orders_messy.csv").exists():
    root = root.parent
orders_path = str(root / "data" / "retail_orders_messy.csv")
customers_path = str(root / "data" / "customers.csv")
assert Path(orders_path).exists(), "Run Jupyter from the repository root, or update orders_path"
orders = spark.read.option("header", True).option("inferSchema", True).csv(orders_path)
customers = spark.read.option("header", True).option("inferSchema", True).csv(customers_path)
print("Raw rows:", orders.count())
orders.printSchema()
orders.show(5, truncate=False)

## Explain plan

In [ ]:
orders.groupBy("region").count().explain(mode="formatted")

## Broadcast small dimension

In [ ]:
from pyspark.sql.functions import broadcast
joined = orders.join(broadcast(customers), "customer_id", "left")
joined.groupBy("segment").count().show()

## Cache and cleanup

In [ ]:
cached = orders.cache()
print("Cached rows:", cached.count())
cached.unpersist()
print("Cache released")

## Reflect

What quality issues did you observe? Which transformations trigger a shuffle? What would you change for production-scale data?

In [ ]:
spark.stop()